# Beginning Python: Become a State Data Detective

In this workshop, we will use a fun dataset about U.S. states to answer questions such as:

- Which states are best for a sunny brunch road trip?
- Which states are UFO hotspots?
- Does warmer weather seem connected to more sunny days?
- Which states are unusual compared with the rest?

We will build these answers with Python, pandas, NumPy, Matplotlib, and SciPy.

**New to coding?** Run each code cell with the play button or Shift+Enter.
Read its result before moving on. You can change a value and run the same
cell again; the notebook is a safe place to experiment.


## Workshop roadmap

| Approx. time | Activity | How we will use it |
|---:|---|---|
| 0–10 min | Python warm-up and notebook setup | Run a cell; change your snack |
| 10–25 min | Explore and filter state data | Select columns; try the UFO filter |
| 25–33 min | NumPy brunch score | Run the instructor demonstration; we need its result later |
| 33–42 min | Make a Matplotlib plot | Change one plot setting yourself |
| 42–47 min | SciPy preview | Short instructor demonstration if time permits |
| 47–57 min | Road-trip champion | Change one preference and compare results |
| 57–60 min | Share a conclusion | Explain what your score can and cannot say |

**Live practice** asks you to change one piece of code at a time. Run a cell
first, see what happens, then make the change and run it again. The extra
examples are here for a demonstration or for exploring after the workshop.


## 1. Setup - Importing Libraries

Libraries are add-ons for Python that allow us to do more cool stuff.

This code imports the libraries, and then gives them a name we will use in our code.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

## 2. A quick Python warm-up

Python can act like a calculator, save information in variables, and call functions.

**Live practice:** Run the next cell. Then change the number and the
snack near the top and run it again. Only edit what is inside the quotes when
changing the snack.


In [ ]:
favorite_number = 6
road_trip_snack = "waffles"

print(f"My favorite number is {favorite_number}.")
print(f"My road-trip snack is {road_trip_snack}.")

6 * 7
"waffles" == "tacos"

## 3. Meet the dataset

Pandas is how we manage data in Python.

A pandas DataFrame is essentially a table. Each row represents a state, and each column describes something about that state.

The dataset has 50 rows and hundreds of columns. We will work with a small
handful of them. The food and animal columns are **search-interest scores**,
not counts of meals eaten or proof that a place welcomes pets.

**First run:** Run the import cell above, then the loading cell below.
If the data does not load, stop here and check the workshop link before
running later examples.


In [ ]:
DATA_URL = "https://hbll.s.gy/pybegin"

# The Python workshop has its own link to the clean workshop CSV.
census_df = pd.read_csv(DATA_URL)
print("Rows and columns:", census_df.shape)

# A few useful columns are easier to read than all 558 at once.
display(census_df[[
    "state", "avg_temp_f", "annual_sunny_days", "capybara",
]].head())
print("Example column names:", census_df.columns[:15].tolist())


### State passports

Select a few columns to create a compact list of states with specific attributes.

**Live practice:** Run the next cell, then replace `"hiking"` in the
list with `"ufo_sightings_per_100k"` and run it again. A column name must
stay inside quotes and must match the data exactly.


In [ ]:
# census_df is the name of our dataframe, and we use the [] to tell it we are looking for data.
# The second set of [] is to create a list of what columns we want.
census_df[[
    "state",
    "personality_profile",
    "avg_temp_f",
    "annual_sunny_days",
    "hiking",
    "capybara",
]]

### Challenge: find a sunny road-trip stop

Find states with at least 200 sunny days per year. Then select only the state name and sunshine columns.

**Hint:** `>= 200` means “at least 200.” In `.loc[rows, columns]`, the
condition goes before the comma and the columns you want go after it.

**Live practice:** Run the completed filter once. Point to the
`200` in the condition and the two column names after the comma. You will
use this same filter pattern for the UFO challenge.


In [ ]:
# I like to think of .loc as location. It basically says where there are more than 200 sunny days, find the state and sunny days info for me.

census_df.loc[
    census_df["annual_sunny_days"] >=200,
    ["state", "annual_sunny_days"],
]

### UFO detectives: filter the data

Find states with at least 60 reported UFO sightings per 100,000 people.
First predict one state that might appear; then run the filter. How many
states did you find? A reported sighting is not evidence of extraterrestrials.

**Live practice:** The next cell supplies the code. Run it, then
change `60` to `70` and run it again. Did fewer states appear? You can put
`60` back afterward.


In [ ]:
ufo_hotspots = census_df[
    census_df["ufo_sightings_per_100k"] >= 60
][[
    "state",
    "ufo_sightings_per_100k",
    "personality_profile",
]]

ufo_hotspots

## 4. NumPy: create a brunch score

NumPy is especially useful when we want to perform the same numerical operation across many rows at once.

First, collect the three brunch columns as a NumPy array.

In the next cell, `trend_columns` is a list of the three column names. The
`.to_numpy()` step turns their values into a rectangular collection of numbers.

**Instructor demonstration:** Run both NumPy code cells in this
section before the road-trip challenge; `brunch_score` is needed there.
Learners only need to predict what adding three food scores will do.


In [ ]:
trend_columns = ["waffles", "pancakes", "donuts"]
trend_array = census_df[trend_columns].to_numpy()

print("Array shape:", trend_array.shape)
print("First state's brunch scores:", trend_array[0])

Use NumPy's sum and mean to calculate a total brunch score and an average brunch score for every state.

`axis=1` means “across each row”: add the three foods for **each state**.
Try predicting whether a state with scores 20, 30, and 40 gets 90.


In [ ]:
census_df = census_df.assign(
    brunch_score=trend_array.sum(axis=1),
    brunch_average=trend_array.mean(axis=1),
)

census_df[[
    "state", "waffles", "pancakes", "donuts",
    "brunch_score", "brunch_average",
]].sort_values("brunch_score", ascending=False).head(10)

### NumPy challenge: define “sunny side”

Use the median number of sunny days as the cutoff. Create a new column called sunshine_label with np.where.

The median splits the states near the middle. `np.where` picks one of two
labels for each row; it does not change the original sunshine values.

**Try later:** The instructor can run this cell to show another
NumPy use. The final road-trip score does not depend on `sunshine_label`.


In [ ]:
sunny_median = np.median(census_df["annual_sunny_days"])

census_df = census_df.assign(
    sunshine_label=np.where(
        census_df["annual_sunny_days"] >= sunny_median,
        "sunny side",
        "cloudier side",
    )
)

census_df[["state", "annual_sunny_days", "sunshine_label"]].head()

## 5. Matplotlib: let the data tell a story

A visualization is useful when it helps us notice a pattern. Start with the distribution of average temperatures.

**Live practice:** Run the histogram in the next cell. Then change
`bins=10` to `bins=5` and run it again. What changed in the picture?


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.hist(census_df["avg_temp_f"], bins=10, edgecolor="white")
ax.set_title("How warm are the states?")
ax.set_xlabel("Average temperature (°F)")
ax.set_ylabel("Number of states")

plt.show()

### Temperature versus sunshine

Each dot is a state. Look for clusters, unusual points, and whether the relationship looks strong or weak.

**Instructor demonstration or try later:** Each dot needs an x
value and a y value. You can return to this example after making your first
plot.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(
    census_df["avg_temp_f"],
    census_df["annual_sunny_days"],
    alpha=0.75,
)

ax.set_title("Temperature and annual sunshine")
ax.set_xlabel("Average temperature (°F)")
ax.set_ylabel("Annual sunny days")

plt.show()

### Personality profiles and sunshine

Group the states by personality profile and make a bar chart of average sunny days.

**Try later:** This chart combines several steps. Focus on the
three profile names and the height of each bar.


In [ ]:
profile_summary = (
    census_df
    .groupby("personality_profile", as_index=False)
    .agg(average_sunny_days=("annual_sunny_days", "mean"))
    .sort_values("average_sunny_days")
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(
    profile_summary["personality_profile"],
    profile_summary["average_sunny_days"],
)
ax.set_title("Which personality profile gets the most sunshine?")
ax.set_ylabel("Average sunny days")
ax.tick_params(axis="x", rotation=45)
plt.show()

## 6. SciPy: find unusual states

A z-score tells us how many standard deviations a value is above or below the average. Large positive values are unusually high; large negative values are unusually low.

We will use UFO reports again here for a *different* question: instead of
filtering above a cutoff, find states that stand out from the others.

**Instructor preview if time permits:** Watch the code find states
that stand out. You do not need to calculate a z-score yourself today.


In [ ]:
ufo_z_scores = stats.zscore(census_df["ufo_sightings_per_100k"])

ufo_outliers = census_df.assign(ufo_z_score=ufo_z_scores)

ufo_outliers[[
    "state", "ufo_sightings_per_100k", "ufo_z_score",
]].sort_values("ufo_z_score", ascending=False).head(5)

### SciPy: quantify the scatterplot

The Pearson correlation measures the direction and strength of a linear relationship. It does not prove that one variable causes the other.

For a first workshop, focus on whether the reported correlation is positive,
negative, or near zero. The p-value is an optional preview of later statistics.

**Instructor preview if time permits:** This measures the pattern
in the scatterplot. It is fine to leave the p-value for another workshop.


In [ ]:
correlation, p_value = stats.pearsonr(
    census_df["avg_temp_f"],
    census_df["annual_sunny_days"],
)

print(f"Correlation: {correlation:.2f}")
print(f"P-value: {p_value:.4f}")

## 7. Final challenge: choose a road-trip champion

You are planning a sunny outdoor brunch road trip with your pet capybara.
A qualifying state must:

- have at least **200** sunny days;
- have at least one national park; and
- have a hiking search-interest score of at least 20.

Create a **practice** `road_trip_score` from brunch, hiking, sunshine, parks,
and a capybara bonus. Because your capybara is coming along, give more weight
to the capybara trend score. Make a separate **yes/no zoo travel rating** so
you can see whether a major zoo is listed; the zoo does not alter the
capybara rating or add points to this trip score. Search interest cannot tell
us whether a place actually allows or is suitable for a pet capybara.

Predict a winner before running the next cell. Which preference would you
change to make a different state win?

**Live practice:** The next cell already contains a working score.
Run it first. Find the line with `capybara_friendly_rating / 2`, change `2`
to `4`, and run it again. Did the top scores change? You can compare the
yes/no zoo rating without changing any score formula.


In [ ]:
# Each & means “and”: a state must pass all three checks.
road_trip_states = census_df[
    (census_df["annual_sunny_days"] >= 200)
    & (census_df["national_parks_count"] >= 1)
    & (census_df["hiking"] >= 20)
].copy()

# These are playful labels for comparing destinations, not measures of safety.
road_trip_states["capybara_friendly_rating"] = road_trip_states["capybara"]
road_trip_states["zoo_travel_rating"] = np.where(
    road_trip_states["has_major_zoo"], "Yes", "No"
)
# LIVE EDIT: after the first run, change the divisor 2 to 4 and rerun.
road_trip_states["capybara_bonus"] = (
    road_trip_states["capybara_friendly_rating"] / 2
)

road_trip_states["road_trip_score"] = (
    road_trip_states["brunch_score"]
    + road_trip_states["hiking"]
    + road_trip_states["annual_sunny_days"] / 10
    + road_trip_states["national_parks_count"] * 5
    + road_trip_states["capybara_bonus"]
)

road_trip_states[[
    "state", "annual_sunny_days", "hiking",
    "national_parks_count", "capybara_friendly_rating",
    "capybara_bonus", "zoo_travel_rating", "road_trip_score",
]].sort_values("road_trip_score", ascending=False)


In [ ]:
top_states = road_trip_states.sort_values("road_trip_score").tail(10)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top_states["state"], top_states["road_trip_score"])
ax.set_title("Road-trip champions")
ax.set_xlabel("Practice road-trip score")

plt.show()

### Share your conclusion

Answer in a sentence or two:

1. Which state would you choose?
2. Which column most influenced your choice?
3. Does your zoo travel rating change your choice, even though it adds no score points?
4. What real-world information would you need before traveling with a pet capybara?

## What you practiced

- Python variables, expressions, and Boolean comparisons
- pandas selection, filtering, assignment, grouping, and sorting
- NumPy arrays, vectorized calculations, medians, and `np.where`
- Matplotlib histograms, scatterplots, and bar charts
- SciPy z-scores and Pearson correlation
- Turning a question into a reproducible analysis